In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

import sys
sys.path.append("../../../utils/")

from utils import *

In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[3]

NOMBRE_DATASET_RESULTADO = "BCCC17__cleanning__v1"

# Ajusta esto si hace falta
RUTA_BASE_RAW = PROJECT_ROOT / "02_datasets" / "raw" / "BCCC17"
RUTA_SALIDA = PROJECT_ROOT / "02_datasets" / "processed_analisis_estadistico" / NOMBRE_DATASET_RESULTADO

NOMBRE_DATASET_LIMPIO = f"{NOMBRE_DATASET_RESULTADO}.csv"
NOMBRE_REPORTE = f"{NOMBRE_DATASET_RESULTADO}_report.json"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"
CORR_THRESHOLD = 0.95
MIN_CLASS_IMPUTE = 50

# ===== COLUMNAS A ELIMINAR MANUALMENTE SI EXISTEN =====
COLUMNAS_A_ELIMINAR = [
    "FLOW_ID",
    "SRC_IP",
    "SRC_PORT",
    "DST_IP"
]

In [3]:
print("Carpeta actual del notebook:")
print(PROJECT_ROOT)
print()
print("Ruta base raw:")
print(Path(RUTA_BASE_RAW).resolve())
print()
print("Ruta salida:")
print(Path(RUTA_SALIDA).resolve())
print()

Carpeta actual del notebook:
/LUSTRE/home/inginf/u32902122/TFG

Ruta base raw:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/raw/BCCC17

Ruta salida:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/BCCC17__cleanning__v1



In [4]:
df = cargar_dataset(
    ruta_base=RUTA_BASE_RAW
)

shape_original = df.shape

print("Forma original del dataset:")
print(shape_original)

df.head()

Forma original del dataset:
(2438052, 122)


,flow_id,timestamp,src_ip,src_port,dst_ip,dst_port,protocol,duration,packets_count,fwd_packets_count,...,bwd_packets_IAT_mean,bwd_packets_IAT_std,bwd_packets_IAT_max,bwd_packets_IAT_min,bwd_packets_IAT_total,subflow_fwd_packets,subflow_bwd_packets,subflow_fwd_bytes,subflow_bwd_bytes,label
0,192.168.10.9_1841_205.174.165.73_8080_TCP_2017...,2017-07-07 09:04:13.990571,192.168.10.9,1841,205.174.165.73,8080,TCP,0.135358,8,4,...,0.044844,0.061008,0.131123,0.001660,0.134532,0.0,0.0,0.0,0.0,Botnet_ARES
1,192.168.10.9_1845_205.174.165.73_8080_TCP_2017...,2017-07-07 09:04:24.131090,192.168.10.9,1845,205.174.165.73,8080,TCP,0.128585,8,4,...,0.042638,0.057734,0.124286,0.001701,0.127915,0.0,0.0,0.0,0.0,Botnet_ARES
2,192.168.10.9_1846_205.174.165.73_8080_TCP_2017...,2017-07-07 09:04:34.262355,192.168.10.9,1846,205.174.165.73,8080,TCP,0.166355,10,5,...,0.041467,0.070571,0.163697,0.000204,0.165869,0.0,0.0,0.0,0.0,Botnet_ARES
3,192.168.10.9_1847_205.174.165.73_8080_TCP_2017...,2017-07-07 09:04:34.526581,192.168.10.9,1847,205.174.165.73,8080,TCP,0.065549,8,4,...,0.021602,0.028732,0.062233,0.001017,0.064805,0.0,0.0,0.0,0.0,Botnet_ARES
4,192.168.10.9_1848_205.174.165.73_8080_TCP_2017...,2017-07-07 09:04:44.594895,192.168.10.9,1848,205.174.165.73,8080,TCP,0.080872,8,4,...,0.026761,0.036088,0.077792,0.000637,0.080284,0.0,0.0,0.0,0.0,Botnet_ARES


In [5]:
df = homogeneizar_columnas(df)

print("Primeras columnas tras homogeneización:")
print(df.columns.tolist()[:20])

Primeras columnas tras homogeneización:
['FLOW_ID', 'TIMESTAMP', 'SRC_IP', 'SRC_PORT', 'DST_IP', 'DST_PORT', 'PROTOCOL', 'DURATION', 'PACKETS_COUNT', 'FWD_PACKETS_COUNT', 'BWD_PACKETS_COUNT', 'TOTAL_PAYLOAD_BYTES', 'FWD_TOTAL_PAYLOAD_BYTES', 'BWD_TOTAL_PAYLOAD_BYTES', 'PAYLOAD_BYTES_MAX', 'PAYLOAD_BYTES_MIN', 'PAYLOAD_BYTES_MEAN', 'PAYLOAD_BYTES_STD', 'PAYLOAD_BYTES_VARIANCE', 'FWD_PAYLOAD_BYTES_MAX']


In [6]:
if LABEL_COL not in df.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL}")

print("Columna objetivo encontrada correctamente.")
print()
print("Distribución inicial de clases:")
display(df[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Columna objetivo encontrada correctamente.

Distribución inicial de clases:


,count
LABEL,
Benign,1786239
DoS_Hulk,349240
Port_Scan,161323
DDoS_LOIT,95733
FTP-Patator,9531
DoS_GoldenEye,8364
DoS_Slowhttptest,6860
SSH-Patator,5949
Botnet_ARES,5508


In [7]:
columnas_presentes_para_eliminar = [c for c in COLUMNAS_A_ELIMINAR if c in df.columns]

df = eliminar_columnas(df, columnas_presentes_para_eliminar)

print("Columnas eliminadas manualmente:")
print(columnas_presentes_para_eliminar)
print()
print("Forma actual:")
print(df.shape)

Columnas eliminadas manualmente:
['FLOW_ID', 'SRC_IP', 'SRC_PORT', 'DST_IP']

Forma actual:
(2438052, 118)


In [8]:
df = limpiar_infinitos_y_vacios(df)

print("Limpieza de infinitos y vacíos completada.")

Limpieza de infinitos y vacíos completada.


In [9]:
filas_antes_duplicados = len(df)

df = eliminar_filas_duplicadas(df)

filas_despues_duplicados = len(df)
duplicados_eliminados = filas_antes_duplicados - filas_despues_duplicados

print("Duplicados eliminados:", duplicados_eliminados)
print("Forma actual:", df.shape)

Duplicados eliminados: 4299
Forma actual: (2433753, 118)


In [10]:
df_sin_nulos, df_con_nulos = separar_filas_con_y_sin_nulos(df)

print("Filas sin nulos:", len(df_sin_nulos))
print("Filas con nulos:", len(df_con_nulos))

Filas sin nulos: 2433753
Filas con nulos: 0


In [11]:
df, filas_imputadas, filas_eliminadas_nulos = imputar_o_eliminar_nulos_por_clase(
    df_sin_nulos=df_sin_nulos,
    df_con_nulos=df_con_nulos,
    label_col=LABEL_COL,
    min_class_impute=MIN_CLASS_IMPUTE
)

print("Filas imputadas:", filas_imputadas)
print("Filas eliminadas por nulos:", filas_eliminadas_nulos)
print("Forma actual:", df.shape)

Filas imputadas: 0
Filas eliminadas por nulos: 0
Forma actual: (2433753, 118)


In [12]:
columnas_antes_constantes = df.shape[1]

df = eliminar_columnas_constantes(df)

columnas_despues_constantes = df.shape[1]
constantes_eliminadas = columnas_antes_constantes - columnas_despues_constantes

print("Columnas constantes eliminadas:", constantes_eliminadas)
print("Forma actual:", df.shape)

Columnas constantes eliminadas: 3
Forma actual: (2433753, 115)


In [13]:
df, columnas_categoricas_codificadas, columnas_ignoradas, columnas_convertidas = codificar_columnas_categoricas_one_hot_seguro(
    df,
    label_col=LABEL_COL,
    max_unique_for_one_hot=50
)

print("Columnas convertidas a numérico:")
print(columnas_convertidas)
print()

print("Columnas codificadas con One-Hot:")
print(columnas_categoricas_codificadas)
print()

print("Columnas ignoradas por alta cardinalidad:")
print(columnas_ignoradas)
print()

print("Forma actual:", df.shape)

Columnas convertidas a numérico:
[]

Columnas codificadas con One-Hot:
['PROTOCOL']

Columnas ignoradas por alta cardinalidad:
['TIMESTAMP']

Forma actual: (2433753, 116)


In [14]:
df = codificar_columnas_string(df)

In [15]:
columnas_antes_corr = df.shape[1]

df = eliminar_columnas_altamente_correlacionadas(
    df,
    threshold=CORR_THRESHOLD,
    label_col=LABEL_COL
)

columnas_despues_corr = df.shape[1]
corr_eliminadas = columnas_antes_corr - columnas_despues_corr

print("Columnas eliminadas por alta correlación:", corr_eliminadas)
print("Forma final tras limpieza:", df.shape)

Columnas eliminadas por alta correlación: 51
Forma final tras limpieza: (2433753, 65)


In [16]:
feature_cols = [c for c in df.columns if c != LABEL_COL]
df = df[feature_cols + [LABEL_COL]]

print("Última columna:", df.columns[-1])
df.head()

Última columna: LABEL


,TIMESTAMP,DST_PORT,DURATION,PACKETS_COUNT,FWD_TOTAL_PAYLOAD_BYTES,PAYLOAD_BYTES_MAX,PAYLOAD_BYTES_MIN,PAYLOAD_BYTES_MEAN,PAYLOAD_BYTES_VARIANCE,FWD_PAYLOAD_BYTES_VARIANCE,...,BWD_CWR_FLAG_COUNTS,BWD_RST_FLAG_COUNTS,PACKETS_IAT_MEAN,FWD_PACKETS_IAT_MEAN,FWD_PACKETS_IAT_STD,BWD_PACKETS_IAT_MEAN,SUBFLOW_FWD_PACKETS,SUBFLOW_FWD_BYTES,PROTOCOL_TCP,LABEL
0,0,8080,0.135358,8,194,194,0,40.25,5132.4375,7056.75,...,0,0,0.019337,0.044937,0.062157,0.044844,0.0,0.0,True,Botnet_ARES
1,1,8080,0.128585,8,194,194,0,40.25,5132.4375,7056.75,...,0,0,0.018369,0.042594,0.058929,0.042638,0.0,0.0,True,Botnet_ARES
2,2,8080,0.166355,10,194,194,0,32.50,4346.6500,6021.76,...,0,0,0.018484,0.041452,0.071034,0.041467,0.0,0.0,True,Botnet_ARES
3,3,8080,0.065549,8,1858,1858,0,248.25,371940.4375,647280.75,...,0,0,0.009364,0.021678,0.030040,0.021602,0.0,0.0,True,Botnet_ARES
4,4,8080,0.080872,8,194,194,0,40.25,5132.4375,7056.75,...,0,0,0.011553,0.026702,0.037218,0.026761,0.0,0.0,True,Botnet_ARES


In [17]:
print("Distribución final de clases:")
display(df[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Distribución final de clases:


,count
LABEL,
Benign,1785043
DoS_Hulk,346201
Port_Scan,161323
DDoS_LOIT,95729
FTP-Patator,9531
DoS_GoldenEye,8364
DoS_Slowhttptest,6856
SSH-Patator,5949
Botnet_ARES,5508


In [18]:
guardar_dataset_csv(
    df=df,
    nombre_archivo=NOMBRE_DATASET_LIMPIO,
    ruta=RUTA_SALIDA
)

print("Dataset limpio guardado correctamente.")
print(Path(RUTA_SALIDA).resolve() / NOMBRE_DATASET_LIMPIO)

Dataset limpio guardado correctamente.
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/BCCC17__cleanning__v1/BCCC17__cleanning__v1.csv
